# 2026.10.14(수) 21h~

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dot, Activation, Concatenate
from tensorflow.keras.callbacks import EarlyStopping

# 문자와 인덱스 매핑
chars = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
num_classes = len(chars)

char_to_index = {char: idx for idx, char in enumerate(chars)}
index_to_char = {idx: char for idx, char in enumerate(chars)}

# 데이터 생성 함수
def generate_data(num_samples):
    X = []
    y = []
    for _ in range(num_samples):
        sequence = np.random.choice(list(chars), 15)
        X.append([char_to_index[char] for char in sequence])
        y.append([char_to_index[char] for char in sequence[::-1]])
    return np.array(X), np.array(y)

# 데이터 생성
num_samples = 20000
X, y = generate_data(num_samples)

# 데이터 형태 변환 (LSTM을 사용하기 위해 3D 텐서로 변환)
X = X.reshape((X.shape[0], X.shape[1], 1))
y = y.reshape((y.shape[0], y.shape[1], 1))

2026-10-14 21:31:23.915645: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
# blocks 쌓기, keras.Sequential(x) functional api 기반(o)
# 인코더 정의
# ------------------------------------------------------------------
# Input(shape=(15, 1)) 의 의미
#  - shape은 "샘플 하나"의 모양이다: 15행 x 1열짜리 2차원 배열.
#    샘플 수(batch) 축은 Keras가 맨 앞에 자동으로 붙여서 실제 모델 입력은 (None, 15, 1)이 된다.
#    (None = 아직 정해지지 않은 샘플 수.) 학습 때는 X.reshape((20000, 15, 1))로 만든
#    (샘플 수, 15, 1) "3차원" 배열이 들어간다. 그래서 "2차원을 입력한다"가 아니라
#    "샘플 하나가 2차원이고, 묶음으로는 3차원"이라고 이해해야 한다.
#  - 15 = 시간 단계(time step): 글자 15개가 순서대로 늘어서 있다. LSTM은 이 축을 따라 한 칸씩 15번 읽는다.
#  - 1  = 특성 수(feature): 한 시점에 들어오는 값의 개수. 글자 하나를 번호(0~51) 하나로 바꿨으니 1개다.
#  - LSTM은 (샘플 수, 시간 단계, 특성 수)의 3차원 입력만 받는다. 글자 번호만 있는
#    (20000, 15) 2차원 배열은 그대로 못 넣어서, 위 데이터 준비에서 reshape(…, 1)로 축을 하나 붙였다.
#  - 샘플 하나의 예) 'C','q','R',… → [[2], [42], [17], …, [..]]   (15행 x 1열)
#  - Embedding을 쓰는 방식(이전 노트북)은 (샘플 수, 15) 2차원 정수를 넣고, Embedding이 각 번호를
#    벡터로 바꿔 3차원으로 만들어 준다. 이 노트북은 Embedding이 없어서 직접 3차원으로 맞춘다.
# ------------------------------------------------------------------
encoder_inputs = Input(shape=(15, 1))
encoder = LSTM(128, return_sequences=True, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs) # 15개의 hidden states
encoder_states = [state_h, state_c] # hidden 및 cell states가 decoder로 전달됨

# 디코더 정의
# 디코더 입력도 같은 모양 (샘플 수, 15, 1). 아래 학습에서는 전부 0인 배열(decoder_input_data)을 넣는다
decoder_inputs = Input(shape=(15, 1))
decoder_lstm = LSTM(128, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states) # 15개의 hidden states

# 어텐션 메커니즘
attention = Dot(axes=[2, 2])([decoder_outputs, encoder_outputs])
attention = Activation('softmax')(attention)
context = Dot(axes=[2, 1])([attention, encoder_outputs])

# 디코더 출력과 컨텍스트 결합
decoder_combined_context = Concatenate(axis=-1)([decoder_outputs, context])

# 출력 레이어
decoder_dense = Dense(num_classes, activation='softmax')
decoder_outputs = decoder_dense(decoder_combined_context)

# 모델 정의
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 입력 데이터와 타겟 데이터를 동일하게 맞춤
decoder_input_data = np.zeros_like(X)

# EarlyStopping 콜백 정의
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 모델 학습
model.fit([X, decoder_input_data], y, epochs=50, validation_split=0.2, callbacks=[early_stopping])

# 모델 검증 함수
def predict_reverse(input_sequence):
    input_sequence = np.array([char_to_index[char] for char in input_sequence]).reshape((1, 15, 1))
    decoder_input = np.zeros((1, 15, 1))
    predicted_sequence = model.predict([input_sequence, decoder_input])
    predicted_indices = np.argmax(predicted_sequence, axis=-1).reshape((15,))
    return ''.join([index_to_char[idx] for idx in predicted_indices])

# 검증용 문자열 리스트 (15글자짜리 30개)
test_strings = ["".join(np.random.choice(list(chars), 15)) for _ in range(30)]

# 실제 거꾸로 된 문자열 리스트
expected_outputs = [s[::-1] for s in test_strings]

# 모델 검증 및 정확도 계산
correct_predictions = 0
total_predictions = len(test_strings)

for i, test_string in enumerate(test_strings):
    predicted_output = predict_reverse(test_string)
    is_correct = predicted_output == expected_outputs[i]
    if is_correct:
        correct_predictions += 1
    print(f"입력: {test_string}")
    print(f"예측된 출력: {predicted_output}")
    print(f"실제 출력: {expected_outputs[i]}")
    print(f"정확 여부: {'맞음' if is_correct else '틀림'}\n")

accuracy = correct_predictions / total_predictions
print(f"총 정확도: {accuracy * 100:.2f}%")


Epoch 1/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.0398 - loss: 3.7965 - val_accuracy: 0.1650 - val_loss: 2.7620
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.2261 - loss: 2.5238 - val_accuracy: 0.3527 - val_loss: 2.0794
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.4068 - loss: 1.9799 - val_accuracy: 0.4859 - val_loss: 1.7263
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.5147 - loss: 1.7237 - val_accuracy: 0.5897 - val_loss: 1.5415
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.6041 - loss: 1.4894 - val_accuracy: 0.6485 - val_loss: 1.3528
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.6703 - loss: 1.3167 - val_accuracy: 0.6863 - val_loss: 1.2185
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.7193 - loss: 1.1892 - val_accuracy: 0.7566 - val_loss: 1.1059
Epoch 8/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.7628 - loss: 1.0848 - val_accu